In [77]:
import pandas as pd
import numpy as np


import xgboost as xgb

from sklearn.metrics import classification_report, roc_auc_score, f1_score

import matplotlib.pyplot as plt

In [79]:
print(xgb.__version__)

3.2.0


In [81]:
df = pd.read_csv("../data/processed/features_2023.csv")

In [83]:
df.head(10)

,Time,LapTime,LapNumber,Stint,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,...,LapTimeDelta,LapTimeRolling3,DegradationFromStintStart,TotalLaps,RacePctComplete,LapsRemaining,IsLateRace,Pitted,Driver,RaceName
0,3845.400,100.625,1.0,1.0,28.883,40.189,35.489,6675.9105,3810.097,3845.518,...,-0.030,100.625000,8.857,58.0,0.017241,57.0,0,0,ALB,Abu Dhabi
1,3938.960,93.560,2.0,1.0,18.892,39.357,35.311,3864.3230,3903.680,3938.991,...,-7.065,97.092500,1.792,58.0,0.034483,56.0,0,0,ALB,Abu Dhabi
2,4030.728,91.768,3.0,1.0,18.588,38.312,34.868,3957.5790,3995.891,4030.759,...,-1.792,95.317667,0.000,58.0,0.051724,55.0,0,0,ALB,Abu Dhabi
3,4122.319,91.591,4.0,1.0,18.657,38.211,34.723,4049.4160,4087.627,4122.350,...,-0.177,92.306333,-0.177,58.0,0.068966,54.0,0,0,ALB,Abu Dhabi
4,4213.741,91.422,5.0,1.0,18.605,38.328,34.489,4140.9550,4179.283,4213.772,...,-0.169,91.593667,-0.346,58.0,0.086207,53.0,0,0,ALB,Abu Dhabi
5,4305.232,91.491,6.0,1.0,18.696,38.325,34.470,4232.4680,4270.793,4305.263,...,0.069,91.501333,-0.277,58.0,0.103448,52.0,0,0,ALB,Abu Dhabi
6,4396.670,91.438,7.0,1.0,18.537,38.553,34.348,4323.8000,4362.353,4396.701,...,-0.053,91.450333,-0.330,58.0,0.120690,51.0,0,0,ALB,Abu Dhabi
7,4488.250,91.580,8.0,1.0,18.473,38.845,34.262,4415.1740,4454.019,4488.281,...,0.142,91.503000,-0.188,58.0,0.137931,50.0,0,0,ALB,Abu Dhabi
8,4579.894,91.644,9.0,1.0,18.449,38.802,34.393,4506.7300,4545.532,4579.925,...,0.064,91.554000,-0.124,58.0,0.155172,49.0,0,0,ALB,Abu Dhabi
9,4671.872,91.978,10.0,1.0,18.548,38.802,34.628,4598.4730,4637.275,4671.903,...,0.334,91.734000,0.210,58.0,0.172414,48.0,0,0,ALB,Abu Dhabi


```
unlike logistic regression, XGBoost doesn't care about feature scale, multicollinearity, or even raw categorical text in some cases, so we use the .csv file rather than the scaled version
```

In [86]:
print(f"Shape of the dataset: {df.shape}")

Shape of the dataset: (11089, 29)


In [88]:
# Split races 

test_races = ["Singapore", "Monza"]

train_df = df[~df["RaceName"].isin(test_races)]
test_df = df[df["RaceName"].isin(test_races)]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (9079, 29)
Test shape: (2010, 29)


In [90]:
# Seperate Features(X) from target (y)

exclude_cols = [
    "Pitted", "Driver", "RaceName", "Compound",
    "IsAccurate", "FastF1Generated", "IsPersonalBest"
]

feature_col = [col for col in train_df.columns if col not in exclude_cols]

X_train = train_df[feature_col]
y_train = train_df["Pitted"]

X_test =test_df[feature_col]
y_test = test_df["Pitted"]


print("Feature columns", feature_col, "\n\n")
print(f"X_train shape: {X_train.shape}\n\n")
print(f"y_train shape: {y_train.shape}")

Feature columns ['Time', 'LapTime', 'LapNumber', 'Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'LapStartTime', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace'] 


X_train shape: (9079, 26)


y_train shape: (9079,)


 ---

```
Time, LapStartTime, Sector1/2/3SessionTime, LapNumber
→ all measuring essentially "race progress" in different units
→ including all of them doesn't help, just adds redundant noise 
  and makes the model slightly slower without real benefit

```

In [92]:
redundant_time_cols = [
    "Time", "LapStartTime",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapNumber",  # superseded by RacePctComplete
    "LapTime"   # superseded by LapTimeRolling3

]

feature_col = [col for col in feature_col if col not in redundant_time_cols]

X_train =train_df[feature_col]
X_test = test_df[feature_col]

print("Cleaned feature columns:", feature_col)
print(f"\nX_train shape: {X_train.shape}")

Cleaned feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

X_train shape: (9079, 19)


In [94]:
# Training the model 
# Handle class imbalance (33 : 1)
# scale_pos_weight tells XGBoost how muh more to weight the rare class

scale_pos_weight = (y_train == 0).sum() /(y_train ==1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

scale_pos_weight: 32.1


In [54]:
model_xgb = xgb.XGBClassifier(
    n_estimators = 300, # no of decision trees to build.
    max_depth = 6, # how deep each individual tree can grow
    learning_rate = 0.05,  # how much each new tree corrects the mistakes of previous trees 
    subsample = 0.8,
    colsample_bytree = 0.8, # each tree only see 80% of the rows+ cols (randomly choosen) this adds randomness helps prevent overfitting
    scale_pos_weight = scale_pos_weight,
    eval_metric = "auc",
    random_state = 42,
    n_jobs = -1

)
model_xgb.fit(X_train, y_train)
print("Model trained")

Model trained


In [96]:
importance = pd.DataFrame({
    "feature": feature_col,
    "Importance": model_xgb.feature_importances_
}).sort_values("Importance", ascending = False)

print(importance)

                      feature  Importance
0                       Stint    0.117725
3                 Sector3Time    0.071343
17              LapsRemaining    0.069793
2                 Sector2Time    0.067229
11            CompoundEncoded    0.064990
8                    TyreLife    0.064734
16            RacePctComplete    0.053470
4                     SpeedI1    0.051877
10                   Position    0.050674
14  DegradationFromStintStart    0.048964
12               LapTimeDelta    0.043861
13            LapTimeRolling3    0.043394
6                     SpeedFL    0.042697
1                 Sector1Time    0.041407
5                     SpeedI2    0.039819
15                  TotalLaps    0.036660
18                 IsLateRace    0.034636
7                     SpeedST    0.032807
9                   FreshTyre    0.023918


In [98]:
for col in ["SpeedFL", "Sector3Time"]:
    print(f"\n{col} by Pitted status:")
    print(df.groupby("Pitted")[col].describe()[["mean", "std", "min", "max"]])


SpeedFL by Pitted status:
              mean        std   min    max
Pitted                                    
0       262.724823  30.732757  59.0  322.0
1       259.252308  32.789043  85.0  311.0

Sector3Time by Pitted status:
             mean       std     min     max
Pitted                                     
0       27.496765  5.042594  20.059  59.997
1       27.123265  4.173735  20.444  44.319


In [100]:
# Predictions

# get predictions on the unseen test races
y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("XGBoost Performance (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred_xgb))

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"ROC-AUC: {auc_xgb:.4f}")

XGBoost Performance (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      1959
           1       0.09      0.10      0.09        51

    accuracy                           0.95      2010
   macro avg       0.53      0.54      0.53      2010
weighted avg       0.95      0.95      0.95      2010

ROC-AUC: 0.7890


```
Why this might be happening:
11. XGBoost might be overfitting to the training races. With max_depth=6 and n_estimators=300, it has a lot of capacity to memorize patterns specific to the 8 training races that don't generalize to Singapore/Monza's different characteristics (street circuit vs. high-speed circuit).


2. The severe class imbalance (33:1) combined with XGBoost's flexibility might cause it to learn overly specific, narrow rules from the training data that don't transfer well — while Logistic Regression's simplicity (one global linear boundary) might actually generalize more robustly here, even though it's "less powerful" in theory.


3. Hyperparameters haven't been tuned at all yet — we used reasonable defaults, but never did any tuning (grid search, cross-validation) to find what actually works best for THIS specific problem

```

 ----

In [104]:
print(type(model_xgb))
print(y_pred_xgb[:20])  # first 20 predictions
print(y_test[:20].values)  # first 20 actual values

<class 'xgboost.sklearn.XGBClassifier'>
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]


In [106]:
# Check 1: Did Pitted accidentally end up IN our features?
print("Pitted" in feature_col)

# Check 2: Are there any features that perfectly correlate with Pitted?

correlations_with_target = train_df[feature_col + ["Pitted"]].corr()["Pitted"].sort_values(ascending=False)
print(correlations_with_target)

False
Pitted                       1.000000
TyreLife                     0.078751
Sector2Time                  0.048546
LapTimeDelta                 0.043954
LapTimeRolling3              0.037647
DegradationFromStintStart    0.028548
LapsRemaining                0.026766
Position                     0.011366
SpeedST                      0.010062
FreshTyre                    0.004001
SpeedI2                      0.002208
SpeedI1                     -0.004659
Sector1Time                 -0.006430
TotalLaps                   -0.007262
Sector3Time                 -0.013595
SpeedFL                     -0.021428
RacePctComplete             -0.032591
IsLateRace                  -0.062311
CompoundEncoded             -0.067462
Stint                       -0.086404
Name: Pitted, dtype: float64


In [108]:
# Are there any duplicate rows between train and test?
overlap = pd.merge(train_df, test_df, how='inner')
print("Number of overlapping rows:", len(overlap))

# Double check race separation is clean
print("Train races:", train_df["RaceName"].unique())
print("Test races:", test_df["RaceName"].unique())
print("Any race appears in both?", 
      bool(set(train_df["RaceName"].unique()) & set(test_df["RaceName"].unique())))

Number of overlapping rows: 0
Train races: ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Saudi Arabia'
 'Silverstone' 'Spain']
Test races: ['Monza' 'Singapore']
Any race appears in both? False


---

In [111]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xgb)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

Confusion Matrix:
[[1907   52]
 [  46    5]]

True Negatives:  1907
False Positives: 52
False Negatives: 46
True Positives:  5


In [113]:
# TrackStatus leaks pit stop information — remove it completely
updated_feature_col = [col for col in feature_col if col != "TrackStatus"]

X_train = train_df[updated_feature_col]
X_test = test_df[updated_feature_col]

print("Updated feature columns:", updated_feature_col)
print(f"\nX_train shape: {X_train.shape}")

Updated feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

X_train shape: (9079, 19)


In [115]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

model_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("XGBoost Performance WITHOUT TrackStatus (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred_xgb))

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"ROC-AUC: {auc_xgb:.4f}")

scale_pos_weight: 32.1
XGBoost Performance WITHOUT TrackStatus (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      1959
           1       0.09      0.10      0.09        51

    accuracy                           0.95      2010
   macro avg       0.53      0.54      0.53      2010
weighted avg       0.95      0.95      0.95      2010

ROC-AUC: 0.7890


In [119]:
f1_scores = []
for threshold in np.arange(0.1, 1.0, 0.01):
    y_pred_thresh = (y_pred_proba_xgb >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    f1_scores.append(f1)


best_idx = np.argmax(f1_scores)
best_threshold_xgb = np.arange(0.1, 1.0, 0.01)[best_idx]
best_f1 = f1_scores[best_idx]
print(f"Best threshold: {best_threshold_xgb:.2f}")
print(f"Best F1: {best_f1:.3f}")


Best threshold: 0.24
Best F1: 0.169
